In [1]:
# 환경 준비

try:                                 #일단 시도 / 오류가 나면 except로 넘어감
    import google.colab              # google colab 전용 도구 가져오기 시도
    IS_COLAB = True
    print("google colab 환경입니다.") # 성공했다면 출력
except ImportError:                  # 오류 발생 시 실행
    IS_COLAB = False
    print("로컬 Jupyter 환경입니다.") # 내 컴퓨터에서 실행중이라고 안내

로컬 Jupyter 환경입니다.


In [2]:
# LangChain 패키지 설치

!pip install -q langchain langchain-openai langchain-core python-dotenv

# ! --> 파이썬 코드 아님, 컴퓨터에 직접 내리는 명령(터미널 명령)
# -q --> 설치과정의 긴 로그를 숨김
# langchain--> Langchain 본체 / langchain-core(프롬프트 템플릿, 파이프 등)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# API키 불러오기

import os         # 운영체제와 소통하는 도구 os 가져옴 / 여기서는 환경변수

if IS_COLAB:      # 앞에서 저장한 것이 참이면 아래 실행
    from google.colab import userdata  #코랩의 비밀보관함 기능 가져옴
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # 키를 꺼내 환경변수에 기록 / 이름이 정확해야함
else :
    from dotenv import load_dotenv
    #python-dotenv에서 파일 읽기 기능 불러옴
    load_dotenv()
    # 같은 폴더의 .env파일을 열어 키를 환경변수에 기록

print("API 키 로드 완료" if os.environ.get("OPENAI_API_KEY") else "키를 못찾았어요")
# 환경변수에서 키를 찾아 있으면 로드완료/ 없으면 못찾음 출력

API 키 로드 완료


In [ ]:
# OpenAI SDK와 LangChain

# 1. OpenAI SDK 방식

from openai import OpenAI   #OpenAI 불러오기

client = OpenAI()           # AI와 대화할 클라이언트 만들기
response = client.chat.completions.create( 
# 채팅방식으로 답변을 만들어달라는 요청 
    model = "gpt-4o-mini",    # 모델선택
    messages=[
        {"role":"system","content": "당신은 데이터 분석가입니다."},
        # system은 배역 지정 / "role"(누가 말하나) / content(어떤 말인가) 
        {"role": "user","content": "이상치 탐지방법을 한 줄로 설명해줘."}
    ])   # user--> 실제 사용자의 질문
print(response.choices[0].message.content)
#답변 보여줌
# choices(후보 답변들) --> [0](첫번째) --> message(메시지)-->content(본문)


print("\n" + "="*50 + "\n") # 줄바꿈


#2 LangChain 방식

from langchain_openai import ChatOpenAI
# LangChain 버전의 OpenAI 연결 부품 ChatOpenAI 가져옴
from langchain_core.prompts import ChatPromptTemplate
#빈칸 채우기 지시문 양식을 만드는 도구 가져옴

llm = ChatOpenAI(model = "gpt-4o-mini")
# 부품 1. 모델

prompt = ChatPromptTemplate.from_messages([ 
# 부품 2. 프롬프트 양식, 메시지 목록으로 양식을 만듬
    ("system","당신은{role}입니다"), # 배역지정 메시지
    ("human","{question}"),        # 사람의 질문 / user = human
])

chain = prompt | llm  
# 부품 3. 체인 / |(파이프)는 왼쪽 결과를 오른쪽으로 넘겨라 라는 뜻
# 양식에 빈칸을 채우고 ai에게 넘기는 라인 완성
result = chain.invoke({"role":"데이터 분석가", "question": "이상치 탐지방법을 한줄로 설명해줘"})
# invoke = 실행, 중괄호{}안에 빈칸의 이름과 채울 값을 짝지어 넘김
print(result.content)
# 답변 출력


print("\n" + "="*50 + "\n") # 줄바꿈


# 3. OpenAI 스타일 딕셔너리

result = llm.invoke([
    {"role":"system","content":"당신은 대학 교수입니다. 학술 용어로 대답하세요"},
    {"role":"user","content": "python을 한줄로 설명해줘"},
])
print("[방식 C]\n", result.content)

이상치 탐지 방법은 데이터 세트에서 일반적인 패턴과 현저히 다른 데이터를 식별하는 기법입니다.


이상치 탐지 방법은 데이터에서 일반적인 패턴과 현저히 다른 값을 식별하여 데이터의 품질을 보장하거나 중요한 통찰력을 도출하는 과정입니다.


In [ ]:
# ChatOpenAI에게 말거는 3가지 방법

# 문자열 1줄

result = llm.invoke("python을 한 줄로 설명해줘")
# llm.invoke() --> AI에게 문장 하나를 그대로 제공, 역할지정 없이 질문만
# result --> AI의 답변을 받은 상자
print("[방식 A 문자열]\n", result.content)
# 제목을 먼저 작성 후 \n으로 줄바꿈 후 답변 본문(.content)를 보여줌

print("\n" + "="*50 + "\n")

# LangChain 메시지 객체

from langchain_core.messages import SystemMessage, HumanMessage
#배역지정 메시지와 사람의 메시지를 만드는 전용도구 가져오기

result = llm.invoke([ #문장 하나 대신 여러개를 목록으로 묶어서 보내기
    SystemMessage("친한 친구처럼 반말로 답변"),
    #AI에게 말투와 태도를 정해주는 메시지
    HumanMessage("python을 한 줄로 설명해줘"),
    # 실제 질문
])
print("[방식 B 메시지 객체]\n", result.content)


print("\n" + "="*50 + "\n")


# 3. OpenAI 스타일 딕셔너리

result = llm.invoke([
    {"role":"system","content":"당신은 대학 교수입니다. 학술 용어로 대답하세요"},
    # sdk 방식 형태 그대로 가져와도 Langchain이 호환해 줌
    # 기존코드 거의 그대로 가져와도 됨
    {"role":"user","content": "python을 한줄로 설명해줘"},
])
print("[방식 C]\n", result.content)



[방식 A 문자열]
 Python은 간결하고 가독성이 높은 문법을 가진 고급 프로그래밍 언어로, 다양한 용도로 사용됩니다.


[방식 B 메시지 객체]
 파이썬은 간결하고 읽기 쉬운 문법을 가진 고급 프로그래밍 언어야.


[방식 C]
 Python은 간결하고 가독성이 높은 문법을 제공하며, 다양한 프로그래밍 패러다임을 지원하는 고급 프로그래밍 언어이다.


In [13]:
# 템플릿 만들기

prompt = ChatPromptTemplate.from_messages([
# ([양식을 만들어 prompt라는 상자에 넣는다])
    ("system", "당신은 {role}입니다. 한국어로 2문장 이내로 대답하세요"),
    #배역 칸은 {role}로 비워두고, "2문장 이내"라는 고정 규칙 추가
    ("human", "{question}"),
    #질문 칸을 {question}으로 비워둠
])
print(prompt)

input_variables=['question', 'role'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['role'], input_types={}, partial_variables={}, template='당신은 {role}입니다. 한국어로 2문장 이내로 대답하세요'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


In [ ]:
# 빈칸만 채워보고 AI는 호출하지 않기

formatted = prompt.invoke({
    # prompt.invoke({ }) --> 양식에 값을 채워넣음 / prompt만 실행해 ai(llm)은 아직 호출 안됨
    "role": "데이터 분석가",
    "question" : "p-value가 뭐야?"
    # role과 question에 이름과 채울 값
})
print("formatted의 타입:", type(formatted)) 
# 결과물의 종류(타입 질문), 메시지 묶음 상자
print()
print(formatted)

formatted의 타입: <class 'langchain_core.prompt_values.ChatPromptValue'>

messages=[SystemMessage(content='당신은 데이터 분석가입니다. 한국어로 2문장 이내로 대답하세요', additional_kwargs={}, response_metadata={}), HumanMessage(content='p-value가 뭐야?', additional_kwargs={}, response_metadata={})]
